In [50]:
!pip install neo4j

In [51]:

from google.colab import drive

print("🧹 Pulizia dei mount precedenti in corso...")
try:
    # Forza la chiusura di qualsiasi connessione zombie e svuota la cache
    drive.flush_and_unmount()
    print("✅ Unmount completato con successo.")
except Exception:
    print("Nessun drive attivo trovato. Procedo...")

print("🔄 Montaggio di Google Drive in corso...")
drive.mount('/content/drive', force_remount=True)
print("✅ Drive pronto!")

🧹 Pulizia dei mount precedenti in corso...
✅ Unmount completato con successo.
🔄 Montaggio di Google Drive in corso...
Mounted at /content/drive
✅ Drive pronto!


In [59]:
from neo4j import GraphDatabase
from google.colab import userdata
# ==========================================
# SETUP NEO4J (RAG) E FUNZIONI CONDIVISE
# ==========================================
print("[INFO] Connessione a Neo4j in corso...")
URI = userdata.get('NEO4J_URI')
USERNAME = userdata.get('NEO4J_USERNAME')
PASSWORD = userdata.get('NEO4J_PASSWORD')

# Crea il driver a livello globale in modo che tutte le celle successive possano usarlo
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
print("[INFO] Connesso a Neo4j!")

[INFO] Connessione a Neo4j in corso...
[INFO] Connesso a Neo4j!


In [139]:
# ==========================================
# FULL VALIDATION PIPELINE (V15 - KG INTEGRATED)
# ==========================================

import json
import os
import re
import collections
from glob import glob
from google.colab import drive

# PATHS
BASE_DIR = "/content/drive/MyDrive/DnD_Project_Data_NLP"
TARGET_DIR = os.path.join(BASE_DIR, "inference_results_base_alpaca")

SUBCLASS_UNLOCK_LEVELS = {
    "cleric": 1, "sorcerer": 1, "warlock": 1,
    "druid": 2, "wizard": 2,
    "default": 3
}

# ==========================================
# UTILS
# ==========================================

def super_normalize(text):
    if not isinstance(text, str): return ""
    return re.sub(r'[^a-z0-9]', '', text.lower())

def has_meaningful_content(val):
    if val is None: return False
    if isinstance(val, str):
        norm = val.strip().lower()
        if norm in ["", "null", "none", "."]: return False
    if isinstance(val, (list, dict)) and len(val) == 0: return False
    if isinstance(val, (int, float)) and val <= 0: return False
    return True

def is_placeholder(val):
    if not has_meaningful_content(val):
        return True
    if isinstance(val, str):
        val_norm = super_normalize(val)
        placeholders = [
            "none", "null", ".", "unknown",
            "noinformationavailable", "_", "-"
        ]
        for ph in placeholders:
            if ph in val_norm:
                return True
    if isinstance(val, list):
        return all(is_placeholder(v) for v in val)
    return False

# ==========================================
# KG FUNCTIONS
# ==========================================

def to_kg_index(name):
    return (name or "").lower().strip().replace(" ", "-")

def kg_get_valid_subclasses(driver, class_name):
    if not class_name: return set()
    query = """
    MATCH (c:Class {index: $class_idx})-[:HAS_SUBCLASS]->(sc:Subclass)
    RETURN collect(sc.name) AS subclasses
    """
    with driver.session() as session:
        res = session.run(query, class_idx=to_kg_index(class_name)).single()
        return set(res["subclasses"]) if res and res["subclasses"] else set()

def kg_get_valid_spells(driver, class_name, subclass_name):
    query = """
    OPTIONAL MATCH (c:Class {index: $class_idx})-[:CAN_LEARN]->(s1:Spell)
    WITH collect(s1.name) AS class_spells

    OPTIONAL MATCH (sc:Subclass {index: $subclass_idx})-[:CAN_LEARN]->(s2:Spell)
    WITH class_spells, collect(s2.name) AS subclass_spells

    RETURN class_spells + subclass_spells AS spells
    """
    with driver.session() as session:
        res = session.run(
            query,
            class_idx=to_kg_index(class_name),
            subclass_idx=to_kg_index(subclass_name)
        ).single()
        return set(res["spells"]) if res and res["spells"] else set()

def kg_get_valid_skills(driver, class_name):
    if not class_name: return set()
    query = """
    MATCH (c:Class {index: $class_idx})-[:CAN_CHOOSE_SKILL]->(sk:Skill)
    RETURN collect(sk.name) AS skills
    """
    with driver.session() as session:
        res = session.run(query, class_idx=to_kg_index(class_name)).single()
        return set(res["skills"]) if res and res["skills"] else set()

def kg_get_class_rules(driver, class_name):
    if not class_name: return {}
    query = """
    MATCH (c:Class {index: $class_idx})
    RETURN c.spellcasting_type AS spell_type
    """
    with driver.session() as session:
        res = session.run(query, class_idx=to_kg_index(class_name)).single()
        return dict(res) if res else {}

# ==========================================
# PARSING LOGIC
# ==========================================

def extract_input_from_prompt(prompt):
    try:
        match = re.search(r'(?:Input Data:|Character Sheet:|### Input:)\s*(\{.*?\})\s*(?:###|$)', prompt, re.DOTALL)
        if match: return json.loads(match.group(1))
    except: pass
    return {}

def detect_repetition_loop(text, threshold=10):
    if len(text) < 100: return False
    tokens = re.split(r'\s+|[,;"]', text)
    tokens = [t for t in tokens if len(t) > 2]
    if not tokens: return False
    counts = collections.Counter(tokens)
    most_common, count = counts.most_common(1)[0]
    if count > threshold:
        if count > 15 and (count / len(tokens) > 0.1): return True
    chunk_size = 30
    if len(text) > chunk_size * 5:
        sub = text[-chunk_size:]
        if text.count(sub) > 5: return True
    return False

def extract_data_fallback(text, input_data=None):
    """
    Fallback parser for JSON and Markdown extraction.
    """
    try:
        start_idx = text.find('{')
        end_idx = text.rfind('}')
        if start_idx != -1 and end_idx != -1:
            json_str = text[start_idx:end_idx+1]
            try:
                data = json.loads(json_str)
            except Exception:
                if json_str.endswith('}}'):
                    json_str = json_str[:-1]
                    data = json.loads(json_str)
                else:
                    raise

            ability_map = {
                "strength": "str", "dexterity": "dex", "constitution": "con",
                "intelligence": "int", "wisdom": "wis", "charisma": "cha",
                "str": "str", "dex": "dex", "con": "con", "int": "int",
                "wis": "wis", "cha": "cha"
            }

            if 'stats' not in data:
                extracted_stats = {}
                keys = list(data.keys())
                for full_name, short_name in ability_map.items():
                    for k in keys:
                        if k.lower() == full_name:
                            extracted_stats[short_name] = data.pop(k)
                if extracted_stats:
                    data['stats'] = extracted_stats

            if input_data:
                for k in ['name', 'race', 'subrace', 'class', 'subclass', 'level', 'background', 'alignment', 'hp', 'ac']:
                    if k not in data and input_data.get(k) is not None:
                        data[k] = input_data[k]
                if 'stats' not in data and 'stats' in input_data:
                    data['stats'] = input_data['stats']

            if 'skills' in data and isinstance(data['skills'], dict):
                data['skills'] = [f"{k}: {v}" for k, v in data['skills'].items()]

            return data, "JSON"
    except Exception:
        pass

    data = {}

    def extract_field(field_names, text_to_search):
        for field in field_names:
            pattern_inline = rf'(?:^|\n)[*#\-\s]*\b{field}\b[*#\s]*:\s*([^\n]+)'
            m1 = re.search(pattern_inline, text_to_search, re.IGNORECASE)
            if m1:
                val = re.sub(r'[*_]', '', m1.group(1)).strip()
                if val and val.lower() not in ['none', 'null', '', 'no information available']:
                    return val
            pattern_header = rf'(?:^|\n)##\s*{field}\s*\n+([^\n]+)'
            m2 = re.search(pattern_header, text_to_search, re.IGNORECASE)
            if m2:
                val = m2.group(1).strip()
                if ':' in val: val = val.split(':', 1)[1]
                val = re.sub(r'[*_]', '', val).strip()
                if val and val.lower() not in ['none', 'null', '', 'no information available']:
                    return val
        return None

    name = extract_field(['Name'], text)
    if name: data['name'] = name
    race = extract_field(['Race'], text)
    if race: data['race'] = race
    subrace = extract_field(['Subrace'], text)
    if subrace: data['subrace'] = subrace

    char_class_raw = extract_field(['Class'], text)
    implicit_subclass = None
    if char_class_raw:
        match = re.search(r'^([A-Za-z\- ]+)\s*\((.+?)\)$', char_class_raw.strip())
        if match:
            data['class'] = match.group(1).strip()
            implicit_subclass = match.group(2).strip()
        else:
            data['class'] = char_class_raw

    subclass = extract_field(['Subclass', 'Archetype', 'Path', 'Tradition', 'Domain', 'Origin'], text)
    if subclass: data['subclass'] = subclass
    elif implicit_subclass: data['subclass'] = implicit_subclass

    level = extract_field(['Level'], text)
    if level and level.isdigit(): data['level'] = int(level)
    bg = extract_field(['Background'], text)
    if bg: data['background'] = bg
    alignment = extract_field(['Alignment'], text)
    if alignment: data['alignment'] = alignment
    hp = extract_field(['HP', 'Hit Points'], text)
    if hp and hp.isdigit(): data['hp'] = int(hp)
    ac = extract_field(['AC', 'Armor Class'], text)
    if ac and ac.isdigit(): data['ac'] = int(ac)

    if input_data:
        for k in ['name', 'race', 'subrace', 'class', 'subclass', 'level', 'background', 'alignment', 'hp', 'ac']:
            if k not in data and input_data.get(k) is not None:
                data[k] = input_data[k]

    ability_map = {
        "strength": "str", "dexterity": "dex", "constitution": "con",
        "intelligence": "int", "wisdom": "wis", "charisma": "cha",
        "str": "str", "dex": "dex", "con": "con", "int": "int",
        "wis": "wis", "cha": "cha"
    }
    data['stats'] = {}
    for line in text.splitlines():
        m = re.search(r'\b(Strength|Dexterity|Constitution|Intelligence|Wisdom|Charisma|Str|Dex|Con|Int|Wis|Cha)\b[^\d]*(\d+)', line, re.I)
        if m:
            key = ability_map[m.group(1).lower()]
            data['stats'][key] = int(m.group(2))

    if input_data and 'stats' in input_data:
        for stat in ['str','dex','con','int','wis','cha']:
            if stat not in data['stats']: data['stats'][stat] = input_data['stats'].get(stat)

    def get_section(header_keywords, text_to_search):
        for kw in header_keywords:
            pattern = rf'(?:(?:^|\n)[*#\s]*\b{kw}\b[*#\s]*:?\s*\n)(.*?)(?=\n\s*\n|\n## |\Z)'
            m = re.search(pattern, text_to_search, re.IGNORECASE | re.DOTALL)
            if m: return m.group(1)
        return ""

    skills_text = get_section(['Skills'], text)
    skills = []
    if skills_text:
        skills_md = re.findall(r'[-\*]\s*\*?\*?\s*([A-Za-z\s]+?)\s*\*?\*?:\s*([+\-]?\d+)', skills_text)
        for skill, val in skills_md: skills.append(f"{skill.strip()}: {val.strip()}")
    if not skills and input_data: skills = input_data.get('skills', [])
    if skills: data['skills'] = skills

    sections_map = {
        'spells': ['Spells', 'Spells Known', 'Spellcasting'],
        'weapons': ['Weapons', 'Attacks'],
        'equipment': ['Equipment', 'Inventory'],
        'feats': ['Feats', 'Features']
    }
    for field, keywords in sections_map.items():
        sec = get_section(keywords, text)
        items = []
        if sec:
            items_md = re.findall(r'[-\*]\s*(.+)', sec)
            for i in items_md:
                clean = re.sub(r'[*_]', '', i).strip()
                if clean: items.append(clean)
        if not items and input_data and field in input_data: items = input_data[field]
        if items: data[field] = items

    return data, "MARKDOWN"

def detect_refusal(raw_text, json_obj=None):
    clean_text = super_normalize(raw_text)
    refusal_keywords = ["cannotgenerate", "unabletofulfill", "cannotcomplete", "invalidhomebrew",
                        "rulebreakingcontent", "inconsistentwithdnd", "safetyguidelines", "violatespolicy",
                        "cannotcreatecharactersheet", "contentpolicy", "cannotcompletethisrequest", "icannot",
                        "sorrybut", "notcontainenoughinformation", "insufficientinformation",
                        "notenoughinformation", "invalidformat", "missingstats"]
    if any(k in clean_text for k in refusal_keywords): return True
    if json_obj:
        if "data" in json_obj and json_obj["data"] is None: return True
        if "message" in json_obj and isinstance(json_obj["message"], str):
            if any(k in super_normalize(json_obj["message"]) for k in refusal_keywords): return True
    return False

# ==========================================
# VALIDATION LOGIC
# ==========================================

def check_mutations(input_json, output_data):
    mutations = []
    for k, v in input_json.items():
        if k in ['stats', 'version']: continue
        if has_meaningful_content(v):
            val_out = output_data.get(k)
            if val_out is None: mutations.append(f"Field '{k}' dropped")
            else:
                if super_normalize(str(val_out)) != super_normalize(str(v)):
                    mutations.append(f"Field '{k}' mutated ({v} -> {val_out})")
    if 'stats' in input_json and isinstance(input_json['stats'], dict):
        out_stats = output_data.get('stats', {})
        for sk, sv in input_json['stats'].items():
            if has_meaningful_content(sv):
                out_sv = out_stats.get(sk)
                if out_sv is None: mutations.append(f"Stat '{sk}' dropped")
                elif str(out_sv) != str(sv): mutations.append(f"Stat '{sk}' mutated")
    return mutations

def check_progress(input_json, output_data):
    added = []
    schema_keys = ['race', 'class', 'subclass', 'background', 'level', 'hp', 'ac', 'alignment', 'spells', 'skills', 'weapons', 'feats']
    for k in schema_keys:
        if not has_meaningful_content(input_json.get(k)) and has_meaningful_content(output_data.get(k)):
            added.append(k)
    input_stats_empty = not has_meaningful_content(input_json.get('stats'))
    if input_stats_empty and len(output_data.get('stats', {})) >= 3:
        added.append('stats')
    return (len(added) > 0), added, input_stats_empty

def validate_game_rules(char_json, input_json, task_type, input_stats_empty, driver):
    errors = []
    passed = []
    class_name = char_json.get('class', 'Unknown')
    class_norm = super_normalize(class_name)
    try: lvl = int(char_json.get('level', 1))
    except: lvl = 1

    # Subclass check
    unlock_lvl = SUBCLASS_UNLOCK_LEVELS.get(class_norm, SUBCLASS_UNLOCK_LEVELS["default"])
    subclass_val = char_json.get('subclass')
    if lvl >= unlock_lvl:
        if not has_meaningful_content(subclass_val):
            if not has_meaningful_content(input_json.get('subclass')): errors.append(f"Missing Subclass for {class_name}")
        else:
            valid_sc = kg_get_valid_subclasses(driver, class_name)
            if valid_sc and super_normalize(subclass_val) not in {super_normalize(s) for s in valid_sc}:
                errors.append(f"Invalid Subclass '{subclass_val}' (KG Check)")

    # HP/AC/Stats
    for field in ['hp', 'ac', 'alignment']:
        if has_meaningful_content(char_json.get(field)):
            if not has_meaningful_content(input_json.get(field)): passed.append(f"Generated {field}")
        elif task_type == 'completion' and not has_meaningful_content(input_json.get(field)):
            errors.append(f"Failed to complete {field}")

    # Spells check
    full_casters = {'bard', 'cleric', 'druid', 'sorcerer', 'warlock', 'wizard'}
    should_have_magic = (class_norm in full_casters) or (lvl >= 2 and class_norm in {'paladin', 'ranger'})
    spells_val = char_json.get('spells', [])
    if should_have_magic and not has_meaningful_content(spells_val) and not has_meaningful_content(input_json.get('spells')):
        errors.append("Missing Spells")
    if has_meaningful_content(spells_val):
        valid_spells = kg_get_valid_spells(driver, class_name, subclass_val)
        valid_norms = {super_normalize(s) for s in valid_spells}
        if valid_norms:
            for sp in spells_val:
                if isinstance(sp, str) and super_normalize(sp) not in valid_norms:
                    errors.append(f"Invalid spell '{sp}' (KG Check)")

    return errors, passed

# ==========================================
# FILE ANALYSIS
# ==========================================

def analyze_predictions(file_path, driver):
    model_name = os.path.basename(file_path).replace("predictions_", "").replace(".json", "")
    print(f"Analyzing {model_name}...")
    try:
        with open(file_path, 'r') as f: predictions = json.load(f)
    except: return None, None, None, None, None

    stats = {"GEN": collections.Counter(), "COMP": collections.Counter(), "REF": collections.Counter()}
    format_stats = {"valid_json": 0, "valid_text_parsed": 0, "garbage": 0, "total": 0}
    log_lines = []

    for idx, entry in enumerate(predictions):
        task_type = entry.get('task_type', 'generation')
        prompt = entry.get('input_prompt', '')
        response = entry.get('generated_response', '')
        input_json = extract_input_from_prompt(prompt)
        is_loop = detect_repetition_loop(response)
        data_obj, source = extract_data_fallback(response)

        format_stats["total"] += 1
        if source == "JSON": format_stats["valid_json"] += 1
        elif source == "MARKDOWN": format_stats["valid_text_parsed"] += 1
        else: format_stats["garbage"] += 1

        status, reasoning, passed_checks = "UNKNOWN", [], []
        is_refusal = detect_refusal(response)

        if task_type == 'refusal':
            if is_refusal:
                status = "SUCCESS"
                stats["REF"]["SUCCESS"] += 1
            else:
                status = "FAILURE"
                stats["REF"]["FAILURE"] += 1
                reasoning.append("Failed to refuse.")
        else:
            cat = "GEN" if task_type == "generation" else "COMP"
            if is_refusal:
                status = "FAIL_REFUSAL"
                stats[cat]["FAIL_REFUSAL"] += 1
            elif data_obj is None:
                status = "FAIL_LOOP" if is_loop else "FAIL_FORMAT"
                stats[cat][status] += 1
            else:
                mutations = check_mutations(input_json, data_obj)
                if mutations:
                    status = "FAIL_MUTATION"
                    stats[cat][status] += 1
                    reasoning.extend(mutations)
                else:
                    progress, fields, stats_empty = check_progress(input_json, data_obj)
                    if not progress:
                        status = "FAIL_STAGNATION"
                        stats[cat][status] += 1
                    else:
                        errs, pass_c = validate_game_rules(data_obj, input_json, task_type, stats_empty, driver)
                        reasoning.extend(errs)
                        passed_checks.extend(pass_c)
                        status = "PARTIAL_SUCCESS" if errs else "SUCCESS"
                        stats[cat][status] += 1

        log_lines.append(f"ENTRY #{idx+1} | TASK: {task_type} | STATUS: {status}\nREASONING: {reasoning}\n")

    return model_name, stats, None, format_stats, log_lines

def generate_report(target_dir_path, driver):
    if not os.path.exists(target_dir_path): return
    files = glob(os.path.join(target_dir_path, "predictions_*.json"))
    report = ["VALIDATION REPORT", "="*40]

    for f in files:
        name, s, _, f_stats, _ = analyze_predictions(f, driver)
        if not name: continue
        report.append(f"\nMODEL: {name}\nJSON Format: {f_stats['valid_json']}/{f_stats['total']}")
        for t in ["GEN", "COMP", "REF"]:
            report.append(f"  {t} results: {dict(s[t])}")

    with open(os.path.join(target_dir_path, "REPORT.txt"), 'w') as out:
        out.write("\n".join(report))
    print("Report generated.")

# Usage:
# generate_report(TARGET_DIR, driver)

In [145]:

# inference on Llama and Qwen

DIR = "/content/drive/MyDrive/DnD_Project_Data_NLP/inference_results_base_alpaca"

#DIR = "/content/drive/MyDrive/DnD_Project_Data_NLP/inference_results_base_chat"

# inference on finetuned Llama and Qwen
#DIR = "/content/drive/MyDrive/DnD_Project_Data_NLP/inference_results_chat"
#DIR = "/content/drive/MyDrive/DnD_Project_Data_NLP/inference_results_alpaca"




generate_report(DIR, driver)


STARTING VALIDATION FOR DIRECTORY: /content/drive/MyDrive/DnD_Project_Data_NLP/inference_results_base_alpaca
Analyzing Llama-3.2-1B-Instruct_Base_Alpaca...
Analyzing Qwen3-0.6B_Base_Alpaca...
Report generated: /content/drive/MyDrive/DnD_Project_Data_NLP/inference_results_base_alpaca/VALIDATION_REPORT.txt


In [146]:
import os
import re
import json
import collections
from glob import glob

# PATHS
BASE_DIR = "/content/drive/MyDrive/DnD_Project_Data_NLP"

REPORT_PATHS = {
    "BASELINE_ALPACA": os.path.join(BASE_DIR, "inference_results_base_alpaca", "VALIDATION_REPORT.txt"),
    "BASELINE_CHAT": os.path.join(BASE_DIR, "inference_results_base_chat", "VALIDATION_REPORT.txt"),
    "FINETUNED_ALPACA": os.path.join(BASE_DIR, "inference_results_alpaca", "VALIDATION_REPORT.txt"),
    "FINETUNED_CHAT": os.path.join(BASE_DIR, "inference_results_chat", "VALIDATION_REPORT.txt"),
}

OUTPUT_MASTER_REPORT = os.path.join(BASE_DIR, "PROJECT_FINAL_COMPARISON_REPORT.txt")

# TEXT DEFINITIONS
METRICS_DEF = """
[METRIC DEFINITIONS]
- Valid JSON: Syntactically correct JSON output.
- GEN Full (Generation Success): Strict adherence to D&D 5e rules (Stats, Spells, Class reqs).
- GEN Part (Generation Partial): Valid JSON with new content but minor rule violations.
- COMP Full (Completion Success): Correct NULL field population without data loss.
- COMP Part (Completion Partial): Fields filled with minor inconsistencies.
- REF Full (Refusal Success): Successful refusal of invalid/unsafe prompts.
"""

ERRORS_DEF = """
[ERROR TYPE DEFINITIONS]
- FAIL_FORMAT_EMPTY: No extractable data or empty JSON.
- FAIL_LOOP: Model output entered a repetition loop.
- FAIL_MUTATION: Hallucinated changes to immutable input data.
- FAIL_STAGNATION: Model echoed input without generating new data.
- FAIL_REFUSAL: Model failed to refuse an invalid request.
- FAIL_COMPLETION: Target fields remained NULL in completion task.
"""

def parse_validation_report(file_path, mode_label):
    if not os.path.exists(file_path):
        print(f"Report not found: {file_path}")
        return []

    with open(file_path, 'r') as f:
        content = f.read()

    sections = content.split("MODEL: ")[1:]
    data = []

    for sec in sections:
        lines = sec.split('\n')
        model_name = lines[0].strip()

        entry = {
            "Model": model_name,
            "Mode": mode_label,
            "Valid JSON": "0.0%",
            "GEN Full": "0.0%",  "GEN Part": "0.0%",
            "COMP Full": "0.0%", "COMP Part": "0.0%",
            "REF Full": "0.0%"
        }

        details = {
            "Model": model_name,
            "Mode": mode_label,
            "GEN": {}, "COMP": {}, "REF": {}
        }

        json_match = re.search(r"Valid JSON:\s+\d+\s+\(([\d\.]+%)\)", sec)
        if json_match: entry["Valid JSON"] = json_match.group(1)

        def get_task_block(text, task_name):
            start = text.find(f"{task_name} TASK")
            return text[start:] if start != -1 else ""

        gen_block = get_task_block(sec, "GEN")
        comp_block = get_task_block(sec, "COMP")
        ref_block = get_task_block(sec, "REF")

        if "COMP TASK" in gen_block: gen_block = gen_block.split("COMP TASK")[0]
        if "REF TASK" in comp_block: comp_block = comp_block.split("REF TASK")[0]

        def extract_stats(block, target_key, data_entry, prefix):
            s_match = re.search(r"-\s+SUCCESS\s+:\s+\d+\s+\(([\d\.]+%)\)", block)
            p_match = re.search(r"-\s+PARTIAL_SUCCESS\s+:\s+\d+\s+\(([\d\.]+%)\)", block)

            data_entry[f"{prefix} Full"] = s_match.group(1) if s_match else "0.0%"
            if prefix != "REF":
                data_entry[f"{prefix} Part"] = p_match.group(1) if p_match else "0.0%"

            all_matches = re.findall(r"-\s+([A-Z_]+)\s+:\s+(\d+)\s+", block)
            for k, v in all_matches:
                details[target_key][k] = int(v)

        extract_stats(gen_block, "GEN", entry, "GEN")
        extract_stats(comp_block, "COMP", entry, "COMP")
        extract_stats(ref_block, "REF", entry, "REF")

        data.append((entry, details))

    return data

# DATA AGGREGATION
all_parsed = []
all_parsed.extend(parse_validation_report(REPORT_PATHS["BASELINE_ALPACA"], "Base (Alpaca)"))
all_parsed.extend(parse_validation_report(REPORT_PATHS["BASELINE_CHAT"], "Base (Chat)"))
all_parsed.extend(parse_validation_report(REPORT_PATHS["FINETUNED_ALPACA"], "FT (Alpaca)"))
all_parsed.extend(parse_validation_report(REPORT_PATHS["FINETUNED_CHAT"], "FT (Chat)"))

table_data = [x[0] for x in all_parsed if x]
details_data = [x[1] for x in all_parsed if x]

mode_order = {"Base (Alpaca)": 0, "Base (Chat)": 1, "FT (Alpaca)": 2, "FT (Chat)": 3}
table_data.sort(key=lambda x: (x['Model'].split('_')[0], mode_order.get(x['Mode'], 99)))
details_data.sort(key=lambda x: (x['Model'].split('_')[0], mode_order.get(x['Mode'], 99)))

# GENERATE MASTER REPORT
report_lines = ["="*100, "                          PROJECT FINAL COMPARISON REPORT", "="*100, ""]
report_lines.append(METRICS_DEF.strip() + "\n\n" + ERRORS_DEF.strip() + "\n\n" + "="*100 + "\n")
report_lines.append("[DETAILED PERFORMANCE BREAKDOWN]\n")

for item in details_data:
    report_lines.append("-" * 80 + f"\nMODEL: {item['Model']} | MODE: {item['Mode']}\n" + "-" * 80)
    for task in ["GEN", "COMP", "REF"]:
        stats = item[task]
        report_lines.append(f"  {task} TASK:")
        sorted_stats = sorted(stats.items(), key=lambda x: x[1], reverse=True)
        for k, v in sorted_stats:
            report_lines.append(f"    - {k:<20}: {v}")
    report_lines.append("")

report_lines.append("="*100 + "\n[FINAL COMPARISON TABLE]")
header_fmt = "| {:<42} | {:<8} | {:<10} | {:<10} | {:<10} | {:<10} | {:<8} |"
divider = "-" * 118
report_lines.append(divider)
report_lines.append(header_fmt.format("MODEL & MODE", "JSON %", "GEN Full", "GEN Part", "COMP Full", "COMP Part", "REF %"))
report_lines.append(divider)

curr_base = ""
for row in table_data:
    m_name = row['Model']
    m_group = "Llama_1B" if "Llama_1B" in m_name else ("Llama_3.2" if "Llama-3.2" in m_name else m_name.split('_')[0])

    if m_group != curr_base and curr_base != "":
        report_lines.append(f"| {'-'*42} | {'-'*8} | {'-'*10} | {'-'*10} | {'-'*10} | {'-'*10} | {'-'*8} |")

    curr_base = m_group
    display_name = f"{row['Model']} [{row['Mode']}]"
    if len(display_name) > 42: display_name = display_name[:39] + ".."

    report_lines.append(header_fmt.format(display_name, row['Valid JSON'], row['GEN Full'], row['GEN Part'], row['COMP Full'], row['COMP Part'], row['REF Full']))

report_lines.append(divider)

with open(OUTPUT_MASTER_REPORT, 'w') as f:
    f.write("\n".join(report_lines))

print(f"Master report saved to: {OUTPUT_MASTER_REPORT}")

# --- CONSOLE PREVIEW ---
print("\n" + "="*118)
print("                                     FINAL COMPARISON PREVIEW")
print("="*118)

header_fmt = "| {:<42} | {:<8} | {:<10} | {:<10} | {:<10} | {:<10} | {:<8} |"
divider = "-" * 118

print(header_fmt.format("MODEL & MODE", "JSON %", "GEN Full", "GEN Part", "COMP Full", "COMP Part", "REF %"))
print(divider)

curr_base_preview = ""

for row in table_data:
    m_name = row['Model']
    # Grouping logic for visual separators
    if "Llama_1B" in m_name:
        m_group = "Llama_1B"
    elif "Llama-3.2" in m_name:
        m_group = "Llama_3.2"
    else:
        m_group = m_name.split('_')[0]

    if m_group != curr_base_preview and curr_base_preview != "":
        print(f"| {'-'*42} | {'-'*8} | {'-'*10} | {'-'*10} | {'-'*10} | {'-'*10} | {'-'*8} |")

    curr_base_preview = m_group

    display_name = f"{row['Model']} [{row['Mode']}]"
    if len(display_name) > 42:
        display_name = display_name[:39] + ".."

    print(header_fmt.format(
        display_name,
        row['Valid JSON'],
        row['GEN Full'],
        row['GEN Part'],
        row['COMP Full'],
        row['COMP Part'],
        row['REF Full']
    ))

print(divider)

Master report saved to: /content/drive/MyDrive/DnD_Project_Data_NLP/PROJECT_FINAL_COMPARISON_REPORT.txt

                                     FINAL COMPARISON PREVIEW
| MODEL & MODE                               | JSON %   | GEN Full   | GEN Part   | COMP Full  | COMP Part  | REF %    |
----------------------------------------------------------------------------------------------------------------------
| Llama-3.2-1B-Instruct_Base_Alpaca [Base..  | 32.2%    | 0.0%       | 26.9%      | 1.4%       | 16.0%      | 0.0%     |
| Llama-3.2-1B-Instruct_Base_Chat [Base (..  | 56.9%    | 0.0%       | 100.0%     | 5.2%       | 88.2%      | 0.0%     |
| Llama-3.2-1B-Instruct_Alpaca_Merged_16b..  | 95.6%    | 0.5%       | 42.5%      | 0.5%       | 61.3%      | 0.0%     |
| Llama-3.2-1B-Instruct_Chat_Merged_16bit..  | 96.2%    | 0.0%       | 1.4%       | 0.0%       | 93.4%      | 0.0%     |
| ------------------------------------------ | -------- | ---------- | ---------- | ---------- | ---------- |